In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lpips

In [3]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import lpips
import matplotlib.pyplot as plt
import time
import numpy as np
import math
import sys

In [4]:
REPO_DIR = '/content/Texture_synthesis'
os.chdir('/content')

if not os.path.exists(REPO_DIR):
  !git clone https://github.com/ValentinaEmili/Texture-synthesis.git Texture_synthesis

if REPO_DIR not in sys.path:
  sys.path.append(REPO_DIR)

Cloning into 'Texture_synthesis'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 169 (delta 75), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 17.68 MiB | 9.23 MiB/s, done.
Resolving deltas: 100% (75/75), done.


In [5]:
!pip install import-ipynb -q
import import_ipynb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.1 MB/s eta 0:00:00


In [6]:
from Texture_synthesis.codebook.VQ_VAE import VQ_VAE

In [7]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])


class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [8]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VQ_VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-5, betas=(0.9, 0.999)) # for VQ-VAE
perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 84.7MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth


## Training

In [ ]:
model.train()
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae'
os.makedirs(save_path, exist_ok=True)
epochs = 40

for epoch in range(epochs):
    epoch_start = time.time()
    total_loss, epoch_perplexity, avg_active_codes = 0.0, 0.0, 0.0
    total_percept, total_recon, total_vq = 0.0, 0.0, 0.0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        data_recon, vq_loss, perplexity, active_codes = model(data)                         # codebook loss

        data = F.interpolate(data, size=(256, 256), mode='bilinear', align_corners=False)
        data_recon = F.interpolate(data_recon, size=(256, 256), mode='bilinear', align_corners=False)

        recon_loss = F.mse_loss(data_recon, data)                                           # reconstruction loss
        percept_loss = perceptual_loss_fn(data_recon, data).mean()                          # perceptual loss
        loss = percept_loss + vq_loss + recon_loss * 0.2
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        epoch_perplexity += perplexity.item()
        avg_active_codes += active_codes

        total_percept += percept_loss
        total_recon += recon_loss
        total_vq += vq_loss
    #visual_validation(model, val_loader, device)
    print(f'====> Epoch: {epoch} | Average loss: {total_loss / len(train_loader):.4f} | Perplexity: {epoch_perplexity/len(train_loader):.2f} | Active codes: {avg_active_codes/len(train_loader):.2f}')
    print(f'----- Perceptual Loss: {total_percept / len(train_loader):.4f} | Reconstruction Loss: {total_recon / len(train_loader):.4f} | Codebook Loss: {total_vq / len(train_loader):.4f} \n')
    # save model
    #epoch_save_path = os.path.join(save_path, f"epoch_{epoch}.pth")
    #torch.save(model.state_dict(), epoch_save_path)


## Validation

In [10]:
def visual_validation(model, val_loader, device):
  model.eval()

  all_indices = []
  visual_samples = None
  total_percept, total_recon, total_vq = 0.0, 0.0, 0.0

  num_embeddings = model.vq.num_embeddings
  embedding_dim = model.vq.embedding_dim
  embeddings = model.vq.embeddings.weight

  with torch.no_grad():
    for batch_idx, (data, _) in enumerate(val_loader):
      data = data.to(device)
      data_recon, vq_loss, _, _ = model(data)

      visual_samples = (data.cpu(), data_recon.cpu())

      data = F.interpolate(data, size=(256, 256), mode='bilinear', align_corners=False)
      data_recon = F.interpolate(data_recon, size=(256, 256), mode='bilinear', align_corners=False)

      recon_loss = F.mse_loss(data_recon, data)                                           # reconstruction loss
      percept_loss = perceptual_loss_fn(data_recon, data).mean()                          # perceptual loss

      total_recon += recon_loss.item()
      total_percept += percept_loss.item()
      total_vq += vq_loss.item()

      z = model.encoder(data)
      z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)

      distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))

      indices = torch.argmin(distances, dim=1)
      all_indices.append(indices.cpu())

  # percentage of used vectors
  encoding_indices = torch.cat(all_indices)
  unique_indices = torch.unique(encoding_indices)
  util_percen = len(unique_indices) / num_embeddings * 100

  # perplexity
  counts = torch.bincount(encoding_indices, minlength=num_embeddings).float()
  probs = counts / counts.sum()
  perplexity = torch.exp(-torch.sum(probs * torch.log(probs + 1e-10)))

  print(f'====> Epoch: {epoch}')

  print(f"Reconstruction Loss: {total_recon / len(val_loader):.4f}")
  print(f"Perceptual Loss: {total_percept / len(val_loader):.4f}")
  print(f"Codebook Loss: {total_vq / len(val_loader):.4f}")

  print(f"Total Codebook Size:    {num_embeddings}")
  print(f"Unique Vectors Used:    {len(unique_indices)} / {num_embeddings}")
  print(f"Codebook Utilization:   {util_percen:.2f}%")
  print(f"Codebook Perplexity:    {perplexity.item():.2f}")

  # visual reconstruction plotting
  real_imgs, recon_imgs = visual_samples
  num_displayed_imgs = min(16, real_imgs.shape[0])

  fig, axes = plt.subplots(2, num_displayed_imgs, figsize=(num_displayed_imgs * 3, 6))

  for i in range(num_displayed_imgs):
    real_img = real_imgs[i].permute(1, 2, 0).numpy()
    recon_img = recon_imgs[i].permute(1, 2, 0).numpy()

    real_plot = ((real_img + 1) / 2).clip(0, 1)
    recon_plot = ((recon_img + 1) / 2).clip(0, 1)

    axes[0, i].imshow(real_plot)
    axes[0, i].set_title(f"original {i+1}")
    axes[0, i].axis('off')

    axes[1, i].imshow(recon_plot)
    axes[1, i].set_title(f"reconstructed {i+1}")
    axes[1, i].axis('off')

  plt.show()

In [ ]:
epochs = 40
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae'

for epoch in range(epochs):
  epoch_save_path = os.path.join(save_path, f"epoch_{epoch}.pth")
  model = VQ_VAE().to(device)
  model.load_state_dict(torch.load(epoch_save_path, weights_only=True))
  visual_validation(model, val_loader, device)



====> Epoch: 0
Reconstruction Loss: 0.0956
Perceptual Loss: 0.5176
Codebook Loss: 1.8955
Total Codebook Size:    512
Unique Vectors Used:    124 / 512
Codebook Utilization:   24.22%
Codebook Perplexity:    30.00

====> Epoch: 1
Reconstruction Loss: 0.1016
Perceptual Loss: 0.5219
Codebook Loss: 2.3291
Total Codebook Size:    512
Unique Vectors Used:    103 / 512
Codebook Utilization:   20.12%
Codebook Perplexity:    27.58

====> Epoch: 2
Reconstruction Loss: 0.0980
Perceptual Loss: 0.5076
Codebook Loss: 2.4035
Total Codebook Size:    512
Unique Vectors Used:    120 / 512
Codebook Utilization:   23.44%
Codebook Perplexity:    30.40

====> Epoch: 3
Reconstruction Loss: 0.0923
Perceptual Loss: 0.5036
Codebook Loss: 2.5569
Total Codebook Size:    512
Unique Vectors Used:    109 / 512
Codebook Utilization:   21.29%
Codebook Perplexity:    29.57

====> Epoch: 4
Reconstruction Loss: 0.0887
Perceptual Loss: 0.4979
Codebook Loss: 2.1508
Total Codebook Size:    512
Unique Vectors Used:    126 / 512
Codebook Utilization:   24.61%
Codebook Perplexity:    30.72

====> Epoch: 5
Reconstruction Loss: 0.0902
Perceptual Loss: 0.4936
Codebook Loss: 2.6205
Total Codebook Size:    512
Unique Vectors Used:    93 / 512
Codebook Utilization:   18.16%
Codebook Perplexity:    30.20

====> Epoch: 6
Reconstruction Loss: 0.0850
Perceptual Loss: 0.4905
Codebook Loss: 2.9888
Total Codebook Size:    512
Unique Vectors Used:    86 / 512
Codebook Utilization:   16.80%
Codebook Perplexity:    29.60

====> Epoch: 7
Reconstruction Loss: 0.0855
Perceptual Loss: 0.4852
Codebook Loss: 2.6214
Total Codebook Size:    512
Unique Vectors Used:    90 / 512
Codebook Utilization:   17.58%
Codebook Perplexity:    30.94

====> Epoch: 8
Reconstruction Loss: 0.0832
Perceptual Loss: 0.4859
Codebook Loss: 2.9925
Total Codebook Size:    512
Unique Vectors Used:    78 / 512
Codebook Utilization:   15.23%
Codebook Perplexity:    30.20

====> Epoch: 9
Reconstruction Loss: 0.0887
Perceptual Loss: 0.4823
Codebook Loss: 2.6926
Total Codebook Size:    512
Unique Vectors Used:    80 / 512
Codebook Utilization:   15.62%
Codebook Perplexity:    31.47

====> Epoch: 10
Reconstruction Loss: 0.0837
Perceptual Loss: 0.4747
Codebook Loss: 2.8031
Total Codebook Size:    512
Unique Vectors Used:    70 / 512
Codebook Utilization:   13.67%
Codebook Perplexity:    31.76

====> Epoch: 11
Reconstruction Loss: 0.0847
Perceptual Loss: 0.4700
Codebook Loss: 2.6174
Total Codebook Size:    512
Unique Vectors Used:    77 / 512
Codebook Utilization:   15.04%
Codebook Perplexity:    32.62

====> Epoch: 12
Reconstruction Loss: 0.0827
Perceptual Loss: 0.4708
Codebook Loss: 2.7382
Total Codebook Size:    512
Unique Vectors Used:    75 / 512
Codebook Utilization:   14.65%
Codebook Perplexity:    31.92

====> Epoch: 13
Reconstruction Loss: 0.0778
Perceptual Loss: 0.4684
Codebook Loss: 3.0163
Total Codebook Size:    512
Unique Vectors Used:    73 / 512
Codebook Utilization:   14.26%
Codebook Perplexity:    32.75

====> Epoch: 14
Reconstruction Loss: 0.0797
Perceptual Loss: 0.4661
Codebook Loss: 2.7981
Total Codebook Size:    512
Unique Vectors Used:    67 / 512
Codebook Utilization:   13.09%
Codebook Perplexity:    32.35

====> Epoch: 15
Reconstruction Loss: 0.0772
Perceptual Loss: 0.4575
Codebook Loss: 2.9515
Total Codebook Size:    512
Unique Vectors Used:    69 / 512
Codebook Utilization:   13.48%
Codebook Perplexity:    32.61

====> Epoch: 16
Reconstruction Loss: 0.0773
Perceptual Loss: 0.4580
Codebook Loss: 2.8073
Total Codebook Size:    512
Unique Vectors Used:    79 / 512
Codebook Utilization:   15.43%
Codebook Perplexity:    33.09

====> Epoch: 17
Reconstruction Loss: 0.0826
Perceptual Loss: 0.4603
Codebook Loss: 2.9001
Total Codebook Size:    512
Unique Vectors Used:    76 / 512
Codebook Utilization:   14.84%
Codebook Perplexity:    33.73

====> Epoch: 18
Reconstruction Loss: 0.0775
Perceptual Loss: 0.4525
Codebook Loss: 2.9261
Total Codebook Size:    512
Unique Vectors Used:    77 / 512
Codebook Utilization:   15.04%
Codebook Perplexity:    33.67

====> Epoch: 19
Reconstruction Loss: 0.0765
Perceptual Loss: 0.4523
Codebook Loss: 2.7954
Total Codebook Size:    512
Unique Vectors Used:    74 / 512
Codebook Utilization:   14.45%
Codebook Perplexity:    33.76

====> Epoch: 20
Reconstruction Loss: 0.0716
Perceptual Loss: 0.4502
Codebook Loss: 2.9865
Total Codebook Size:    512
Unique Vectors Used:    68 / 512
Codebook Utilization:   13.28%
Codebook Perplexity:    33.81

====> Epoch: 21
Reconstruction Loss: 0.0737
Perceptual Loss: 0.4487
Codebook Loss: 2.8072
Total Codebook Size:    512
Unique Vectors Used:    85 / 512
Codebook Utilization:   16.60%
Codebook Perplexity:    34.22

====> Epoch: 22
Reconstruction Loss: 0.0708
Perceptual Loss: 0.4492
Codebook Loss: 2.8434
Total Codebook Size:    512
Unique Vectors Used:    78 / 512
Codebook Utilization:   15.23%
Codebook Perplexity:    33.92

====> Epoch: 23
Reconstruction Loss: 0.0701
Perceptual Loss: 0.4459
Codebook Loss: 3.0147
Total Codebook Size:    512
Unique Vectors Used:    72 / 512
Codebook Utilization:   14.06%
Codebook Perplexity:    33.35

====> Epoch: 24
Reconstruction Loss: 0.0714
Perceptual Loss: 0.4432
Codebook Loss: 2.9452
Total Codebook Size:    512
Unique Vectors Used:    77 / 512
Codebook Utilization:   15.04%
Codebook Perplexity:    34.21

====> Epoch: 25
Reconstruction Loss: 0.0707
Perceptual Loss: 0.4414
Codebook Loss: 2.7794
Total Codebook Size:    512
Unique Vectors Used:    74 / 512
Codebook Utilization:   14.45%
Codebook Perplexity:    34.26

====> Epoch: 26
Reconstruction Loss: 0.0698
Perceptual Loss: 0.4388
Codebook Loss: 2.8506
Total Codebook Size:    512
Unique Vectors Used:    86 / 512
Codebook Utilization:   16.80%
Codebook Perplexity:    35.33

====> Epoch: 27
Reconstruction Loss: 0.0756
Perceptual Loss: 0.4398
Codebook Loss: 2.8779
Total Codebook Size:    512
Unique Vectors Used:    87 / 512
Codebook Utilization:   16.99%
Codebook Perplexity:    35.05

====> Epoch: 28
Reconstruction Loss: 0.0679
Perceptual Loss: 0.4353
Codebook Loss: 3.0226
Total Codebook Size:    512
Unique Vectors Used:    77 / 512
Codebook Utilization:   15.04%
Codebook Perplexity:    35.03

====> Epoch: 29
Reconstruction Loss: 0.0714
Perceptual Loss: 0.4349
Codebook Loss: 3.0038
Total Codebook Size:    512
Unique Vectors Used:    72 / 512
Codebook Utilization:   14.06%
Codebook Perplexity:    34.61

====> Epoch: 30
Reconstruction Loss: 0.0692
Perceptual Loss: 0.4347
Codebook Loss: 2.9616
Total Codebook Size:    512
Unique Vectors Used:    78 / 512
Codebook Utilization:   15.23%
Codebook Perplexity:    35.06

====> Epoch: 31
Reconstruction Loss: 0.0739
Perceptual Loss: 0.4347
Codebook Loss: 2.9119
Total Codebook Size:    512
Unique Vectors Used:    86 / 512
Codebook Utilization:   16.80%
Codebook Perplexity:    34.91

====> Epoch: 32
Reconstruction Loss: 0.0682
Perceptual Loss: 0.4307
Codebook Loss: 2.8964
Total Codebook Size:    512
Unique Vectors Used:    90 / 512
Codebook Utilization:   17.58%
Codebook Perplexity:    35.97

====> Epoch: 33
Reconstruction Loss: 0.0700
Perceptual Loss: 0.4294
Codebook Loss: 2.8521
Total Codebook Size:    512
Unique Vectors Used:    89 / 512
Codebook Utilization:   17.38%
Codebook Perplexity:    35.31

====> Epoch: 34
Reconstruction Loss: 0.0687
Perceptual Loss: 0.4286
Codebook Loss: 2.6187
Total Codebook Size:    512
Unique Vectors Used:    99 / 512
Codebook Utilization:   19.34%
Codebook Perplexity:    36.11

====> Epoch: 35
Reconstruction Loss: 0.0650
Perceptual Loss: 0.4297
Codebook Loss: 2.8500
Total Codebook Size:    512
Unique Vectors Used:    83 / 512
Codebook Utilization:   16.21%
Codebook Perplexity:    36.48

====> Epoch: 36
Reconstruction Loss: 0.0681
Perceptual Loss: 0.4297
Codebook Loss: 2.4762
Total Codebook Size:    512
Unique Vectors Used:    105 / 512
Codebook Utilization:   20.51%
Codebook Perplexity:    35.82

====> Epoch: 37
Reconstruction Loss: 0.0658
Perceptual Loss: 0.4235
Codebook Loss: 2.6953
Total Codebook Size:    512
Unique Vectors Used:    95 / 512
Codebook Utilization:   18.55%
Codebook Perplexity:    36.05

====> Epoch: 38
Reconstruction Loss: 0.0704
Perceptual Loss: 0.4252
Codebook Loss: 2.6680
Total Codebook Size:    512
Unique Vectors Used:    94 / 512
Codebook Utilization:   18.36%
Codebook Perplexity:    36.72